In [5]:
# ---------------- Imports ----------------
import os
import json
import random
from pathlib import Path
import yaml

In [6]:
# ---------------- Args ----------------
dataset_name = "yield-v1"



In [7]:
# ---------------- Config ----------------
with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

proj_store = config["paths"]["proj_store"]
data_path = os.path.join(proj_store, "data")

input_dir = os.path.join(data_path, dataset_name)
output_dir = os.path.join(data_path, f"{dataset_name}-shuffled")
os.makedirs(output_dir, exist_ok=True)






In [8]:
# ---------------- Main ----------------



def shuffle_utterances_within_dialogues(data, seed=None):
    if seed is not None:
        random.seed(seed)

    for dialogue in data:
        turns = dialogue.get("turns", [])
        utterances = [turn["utterance"] for turn in turns]
        random.shuffle(utterances)

        for turn, new_utt in zip(turns, utterances):
            turn["utterance"] = new_utt

    return data


def process_dataset(input_root, output_root, seed=None):
    input_root = Path(input_root)
    output_root = Path(output_root)

    for json_path in input_root.rglob("*.json"):
        # Preserve relative path
        relative_path = json_path.relative_to(input_root)
        output_path = output_root / relative_path

        # Create output subfolders if needed
        output_path.parent.mkdir(parents=True, exist_ok=True)

        # Load, process, save
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        shuffled_data = shuffle_utterances_within_dialogues(data, seed=seed)

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(shuffled_data, f, ensure_ascii=False, indent=2)


# Example usage
process_dataset(
    input_root=input_dir,
    output_root=output_dir,
    seed=42
)

